# Section 3 - Transformer-Based Modelling (BERT)

Base model: google-bert/bert-base-uncased

Runs (all share identical settings and only the training data changes):

- A_general_only - Model A (reference): TweetEval general tweets only, scored on the finance test set to show what happens with no finance data
- S1_domain_only - Strategy 1: StockEmotions finance posts only
- S2_sequential - Strategy 2: TweetEval first and then continued training on StockEmotions
- S3_mixed - Strategy 3: TweetEval + StockEmotions combined in one training run

Every run is scored on stock_emotions_test.csv (1,000 posts) and repeated with seeds 42, 43 and 44.

How to run

1. Runtime -> Change runtime type -> T4 GPU.
2. In Google Drive create MyDrive/COS60011_BERT/data/ and copy the 6 CSV files from src/pre processing/data/processed/ into it (tweet_eval_train, tweet_eval_validation, stock_emotions_train, stock_emotions_validation, stock_emotions_test, mixed_train).
3. Runtime -> Run all. The full run takes about 2 hours on a T4. For a 10 minute test set QUICK_TEST = True in the settings cell (200 rows per file, 1 epoch, outputs go to separate _quick folders).

Outputs (saved to Drive COS60011_BERT/)

- results/bert_results.csv - one row per run (accuracy, precision, recall, F1, macro-F1, bearish recall, confusion matrix counts)
- results/images/ - confusion matrix per run + bert_strategy_comparison.png
- results/predictions/ - per-post predictions for every run
- results/logs/ - training history per run
- models/ - saved models (seed 42 only for S1/S2/S3, plus the general models needed by Strategy 2)

## 1. Environment check

In [9]:
import sys
import torch
import transformers
import accelerate
import datasets
import sklearn
import pandas

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4 GPU")
print("Python:", sys.version)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)

GPU available: True
GPU name: Tesla T4
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch: 2.11.0+cu130
transformers: 5.20.2
accelerate: 1.15.0
datasets: 4.8.5
scikit-learn: 1.6.1
pandas: 2.2.3


## 2. Install the one extra package (emoji to words)

In [10]:
!pip install -q emoji

## 3. Mount Google Drive

In [11]:
from google.colab import drive
drive.mount("/content/drive")

import os

BASE_DIR = "/content/drive/MyDrive/COS60011_BERT"
DATA_DIR = BASE_DIR + "/data"
CHECKPOINT_DIR = "/content/checkpoints"   # temporary training checkpoints (not on Drive)

assert os.path.isdir(DATA_DIR), "Create MyDrive/COS60011_BERT/data in Google Drive and copy the 6 CSV files into it"
print(sorted(os.listdir(DATA_DIR)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['mixed_train.csv', 'stock_emotions_test.csv', 'stock_emotions_train.csv', 'stock_emotions_validation.csv', 'tweet_eval_train.csv', 'tweet_eval_validation.csv']


## 4. Imports

In [12]:
import gc
import json
import time
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import emoji

from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

## 5. Settings (shared by every run)

Only the training data changes between strategies and everything below stays fixed.

In [13]:
MODEL_NAME = "google-bert/bert-base-uncased"
MAX_LENGTH = 128          # tokens per post
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EPOCHS = 4
WARMUP = 0.1
WEIGHT_DECAY = 0.01

ID2LABEL = {0: "bearish/negative", 1: "bullish/positive"}
LABEL2ID = {"bearish/negative": 0, "bullish/positive": 1}

SEEDS = [42, 43, 44]      # every strategy is repeated 3 times
USE_EMOJI_WORDS = True    # convert emoji to words since BERT's vocabulary has no emoji
CMAP = "Blues"            # colour for the confusion matrices

QUICK_TEST = False        # True = fast test run: 200 rows per file, 1 epoch, outputs go to *_quick folders on drive
UPLOAD_TO_HUB = False     # True = push the seed-42 models to Hugging Face at the end (needs a Colab secret HF_TOKEN)

if QUICK_TEST:
    EPOCHS = 1
RUN_TAG = "_quick" if QUICK_TEST else ""
MODELS_DIR = BASE_DIR + "/models" + RUN_TAG
RESULTS_DIR = BASE_DIR + "/results" + RUN_TAG
for folder in [RESULTS_DIR + "/images", RESULTS_DIR + "/predictions", RESULTS_DIR + "/logs", MODELS_DIR, CHECKPOINT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Settings loaded | QUICK_TEST =", QUICK_TEST, "| EPOCHS =", EPOCHS, "| SEEDS =", SEEDS)
print("Results folder:", RESULTS_DIR)

Settings loaded | QUICK_TEST = False | EPOCHS = 4 | SEEDS = [42, 43, 44]
Results folder: /content/drive/MyDrive/COS60011_BERT/results


## 6. Text preparation

Two fixes before tokenising: literal \uXXXX escape codes left in TweetEval are turned back into characters,
and emoji are turned into words (e.g. 🚀 -> rocket) because bert-base-uncased has no emoji in its vocabulary.

In [14]:
def prepare_text(text):
    text = str(text)
    text = re.sub(r"\\u([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), text)
    if USE_EMOJI_WORDS:
        text = emoji.demojize(text, delimiters=(" ", " ")).replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(prepare_text("$TSLA to the moon🚀🚀 can\\u2019t stop"))

$TSLA to the moon rocket rocket can’t stop


## 7. Data loading

In [ ]:
def load_data(file_name):
    df = pd.read_csv(DATA_DIR + "/" + file_name)
    keep = [c for c in ["text", "label", "emotion", "ticker"] if c in df.columns]
    df = df[keep].copy()
    df["text"] = df["text"].astype(str).apply(prepare_text)
    df["label"] = df["label"].astype(int)
    if QUICK_TEST:
        df = df.sample(n=200, random_state=42)
    return df.reset_index(drop=True)

## 8. Load all datasets

In [ ]:
general_train = load_data("tweet_eval_train.csv")
general_val   = load_data("tweet_eval_validation.csv")
finance_train = load_data("stock_emotions_train.csv")
finance_val   = load_data("stock_emotions_validation.csv")
finance_test  = load_data("stock_emotions_test.csv")
mixed_train   = load_data("mixed_train.csv")

for name, df in [("general_train", general_train), ("general_val", general_val),
                 ("finance_train", finance_train), ("finance_val", finance_val),
                 ("finance_test", finance_test), ("mixed_train", mixed_train)]:
    print(f"{name:14} rows={len(df):6}  label0={int((df['label']==0).sum()):6}  label1={int((df['label']==1).sum()):6}")

print()
print(finance_train["text"].head(3).to_list())

## 9. Tokenizer check

How BERT splits the text and how long posts are...

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

examples = finance_train["text"].head(3).to_list() + general_train["text"].head(3).to_list()
for t in examples:
    print(tokenizer.tokenize(t))
print()

def token_lengths(df):
    enc = tokenizer(df["text"].to_list(), truncation=False)
    return np.array([len(ids) for ids in enc["input_ids"]])

for name, df in [("finance_train", finance_train), ("general_train", general_train), ("finance_test", finance_test)]:
    lengths = token_lengths(df)
    print(f"{name:14} p95={np.percentile(lengths, 95):.0f}  p99={np.percentile(lengths, 99):.0f}  "
          f"max={lengths.max()}  over_{MAX_LENGTH}={int((lengths > MAX_LENGTH).sum())}")

## 10. DataFrame to tokenised dataset

In [ ]:
def make_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    ds = ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH), batched=True)
    return ds

tiny = make_dataset(finance_val.head(5))
print(tiny)
print(tiny[0]["input_ids"][:10])

## 11. Metrics
Same ones as the Section 2 baselines plus macro-F1 and bearish recall

In [ ]:
from transformers import EvalPrediction

def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "bearish_recall": recall_score(labels, preds, pos_label=0, zero_division=0),
    }

fake = EvalPrediction(predictions=np.array([[2.0, 0.1], [0.1, 2.0], [2.0, 0.1], [0.1, 2.0]]), label_ids=np.array([0, 1, 1, 1]))
print(compute_metrics(fake))

## 12. Training function

Trains BERT with the shared settings, keeps the epoch with the best validation macro-F1, and saves the training log.
start_from is either the base model name or a folder with an already fine-tuned model (used by Strategy 2).

In [ ]:
def train_bert(start_from, train_df, val_df, run_name, seed, save_to=None):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        start_from, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)

    train_ds = make_dataset(train_df)
    val_ds = make_dataset(val_df)

    args = TrainingArguments(
        output_dir=CHECKPOINT_DIR + "/" + run_name,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        warmup_steps=WARMUP,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=False,     # reloaded manually below (transformers 5 reload bug)
        greater_is_better=True,
        metric_for_best_model="macro_f1",
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none",
        seed=seed,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60

    # Reload the best epoch.
    best_dir = trainer.state.best_model_checkpoint
    best_model = AutoModelForSequenceClassification.from_pretrained(best_dir)
    trainer.model.load_state_dict(best_model.state_dict(), strict=True)
    del best_model
    print("best epoch reloaded from", best_dir)

    with open(RESULTS_DIR + "/logs/" + run_name + ".json", "w") as f:
        json.dump(trainer.state.log_history, f, indent=1)

    if save_to is not None:
        trainer.save_model(save_to)
        tokenizer.save_pretrained(save_to)

    print(f"{run_name}: {minutes:.1f} min | best validation macro_f1 = {trainer.state.best_metric:.4f}")
    return trainer, minutes

## 13. Evaluation function

Scores a trained model on the finance test set, saves the confusion matrix image, the per-post predictions,
and adds one row to bert_results.csv (a re-run of the same strategy and seed replaces its old row).

In [ ]:
def evaluate_bert(trainer, test_df, strategy, seed, n_train, minutes):
    test_ds = make_dataset(test_df)
    output = trainer.predict(test_ds)

    logits = output.predictions.astype(np.float32)
    labels = output.label_ids
    preds = np.argmax(logits, axis=-1)
    prob_bullish = softmax(logits, axis=1)[:, 1]

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = [int(x) for x in cm.ravel()]

    row = {
        "strategy": strategy, "seed": seed, "n_train": n_train, "minutes": round(minutes, 2),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "bearish_recall": recall_score(labels, preds, pos_label=0, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "best_val_macro_f1": trainer.state.best_metric,
    }

    ConfusionMatrixDisplay(cm, display_labels=["bearish", "bullish"]).plot(cmap=CMAP, colorbar=False)
    plt.title(f"BERT {strategy} (seed {seed})")
    plt.savefig(f"{RESULTS_DIR}/images/{strategy}_seed{seed}.png", dpi=150, bbox_inches="tight")
    plt.close()

    pred_df = test_df.copy()
    pred_df["pred"] = preds
    pred_df["prob_bullish"] = prob_bullish
    pred_df["correct"] = (pred_df["pred"] == pred_df["label"]).astype(int)
    pred_df.to_csv(f"{RESULTS_DIR}/predictions/{strategy}_seed{seed}.csv", index=False)

    results_path = RESULTS_DIR + "/bert_results.csv"
    if os.path.exists(results_path):
        results = pd.read_csv(results_path)
        results = results[~((results["strategy"] == strategy) & (results["seed"] == seed))]   # drop old row for this run
        results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    else:
        results = pd.DataFrame([row])
    results.to_csv(results_path, index=False)

    print(f"{strategy} seed {seed}: accuracy={row['accuracy']:.3f}  macro_f1={row['macro_f1']:.3f}  "
          f"bearish_recall={row['bearish_recall']:.3f}  confusion=[[{tn}, {fp}], [{fn}, {tp}]]")
    return row

## 14. GPU memory cleanup

This function is called between runs.

In [ ]:
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared")

## 15. Run 1 - general tweets only (Model A, and stage 1 of Strategy 2)

Fresh BERT trained on TweetEval. Scored on the finance test set as **Model A** (no finance data at all).
The saved model is the starting point for Strategy 2.

In [ ]:
for seed in SEEDS:
    trainer, minutes = train_bert(MODEL_NAME, general_train, general_val, f"s2_stage1_seed{seed}", seed,
                                  save_to=f"{MODELS_DIR}/general_seed{seed}")
    evaluate_bert(trainer, finance_test, "A_general_only", seed, len(general_train), minutes)
    del trainer
    cleanup()

## 16. Run 2 - Strategy 2 (sequential): general model to finance tweets

Continues training the saved general model on StockEmotions. Its cost is Run 1 + Run 2, so the minutes are added together.

In [ ]:
for seed in SEEDS:
    trainer, minutes = train_bert(f"{MODELS_DIR}/general_seed{seed}", finance_train, finance_val,
                                  f"s2_seq_seed{seed}", seed,
                                  save_to=f"{MODELS_DIR}/s2_seed{seed}" if seed == 42 else None)
    results = pd.read_csv(RESULTS_DIR + "/bert_results.csv")
    run1_minutes = results[(results["strategy"] == "A_general_only") & (results["seed"] == seed)]["minutes"].iloc[0]
    evaluate_bert(trainer, finance_test, "S2_sequential", seed,
                  len(general_train) + len(finance_train), minutes + run1_minutes)
    del trainer
    cleanup()

## 17. Run 3 - Strategy 1 (domain only): fresh BERT on finance tweets only

In [ ]:
for seed in SEEDS:
    trainer, minutes = train_bert(MODEL_NAME, finance_train, finance_val, f"s1_seed{seed}", seed,
                                  save_to=f"{MODELS_DIR}/s1_seed{seed}" if seed == 42 else None)
    evaluate_bert(trainer, finance_test, "S1_domain_only", seed, len(finance_train), minutes)
    del trainer
    cleanup()

## 18. Run 4 - Strategy 3 (mixed): fresh BERT on general + finance combined

In [ ]:
for seed in SEEDS:
    trainer, minutes = train_bert(MODEL_NAME, mixed_train, finance_val, f"s3_seed{seed}", seed,
                                  save_to=f"{MODELS_DIR}/s3_seed{seed}" if seed == 42 else None)
    evaluate_bert(trainer, finance_test, "S3_mixed", seed, len(mixed_train), minutes)
    del trainer
    cleanup()

## 19. Results of every run

In [ ]:
results = pd.read_csv(RESULTS_DIR + "/bert_results.csv").sort_values(["strategy", "seed"])
cols = ["strategy", "seed", "n_train", "minutes", "accuracy", "precision", "recall", "f1", "macro_f1", "bearish_recall", "best_val_macro_f1"]
print(results[cols].round(3).to_string(index=False))

## 20. Strategy Comparison on Results

Macro-F1 is the headline metric because the test set is imbalanced (555 bullish / 445 bearish) and it weights both classes equally.
The dashed line is the best TF-IDF baseline from Section 2 (Logistic Regression, Model B, macro-F1 0.711).

In [ ]:
BASELINE_MACRO_F1 = 0.711   # best Section 2 baseline (TF-IDF + Logistic Regression, Model B)
ORDER = ["A_general_only", "S1_domain_only", "S2_sequential", "S3_mixed"]
NAMES = {"A_general_only": "Model A\ngeneral only", "S1_domain_only": "Strategy 1\nfinance only",
         "S2_sequential": "Strategy 2\nsequential", "S3_mixed": "Strategy 3\nmixed"}

summary = results.groupby("strategy")[["accuracy", "macro_f1", "bearish_recall", "minutes"]].agg(["mean", "std"]).round(3)
summary = summary.reindex([s for s in ORDER if s in summary.index])
print(summary.to_string())
summary.to_csv(RESULTS_DIR + "/bert_strategy_summary.csv")

means = summary[("macro_f1", "mean")]
stds = summary[("macro_f1", "std")].fillna(0)
plt.figure(figsize=(7, 4))
bars = plt.bar([NAMES[s] for s in means.index], means, yerr=stds, capsize=4, color="#2171b5", width=0.6)
for bar, value in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.3f}", ha="center", fontsize=9)
plt.axhline(BASELINE_MACRO_F1, linestyle="--", color="gray", linewidth=1)
plt.text(-0.3, BASELINE_MACRO_F1 + 0.012, f"best TF-IDF baseline ({BASELINE_MACRO_F1})", ha="left", fontsize=8, color="gray")
plt.ylim(0, 1)
plt.ylabel("Macro-F1 on finance test set")
plt.title("BERT training strategies (mean ± std over seeds 42, 43, 44)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR + "/images/bert_strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 21. Write files into Drive

In [ ]:
for folder in ["", "/images", "/predictions", "/logs"]:
    print("results" + folder, sorted(os.listdir(RESULTS_DIR + folder)))
print("models", sorted(os.listdir(MODELS_DIR)))

## 22. Extra Enhancement - Emotion Model (12 StockEmotions emotion labels)

Not part of the three-strategy comparison. A second BERT trained on the StockEmotions emotion column
(same settings, seed 42, finance data only) so the Streamlit demo can show an emotion next to the sentiment.

In [ ]:
EMOTIONS = sorted(pd.concat([finance_train, finance_val, finance_test])["emotion"].unique())
EMO2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2EMO = {i: e for e, i in EMO2ID.items()}
print(len(EMOTIONS), "emotions:", EMOTIONS)

def emotion_frame(df):
    out = df[["text"]].copy()
    out["label"] = df["emotion"].map(EMO2ID).astype(int)
    return out

emo_train, emo_val, emo_test = emotion_frame(finance_train), emotion_frame(finance_val), emotion_frame(finance_test)
print("train rows per emotion:", emo_train["label"].value_counts().sort_index().to_list())

def emotion_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

set_seed(42)
emo_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(EMOTIONS), id2label=ID2EMO, label2id=EMO2ID)

emo_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR + "/emotion_seed42",
    learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE, num_train_epochs=EPOCHS,
    warmup_steps=WARMUP, weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=False,
    metric_for_best_model="macro_f1", greater_is_better=True, save_total_limit=1,
    logging_strategy="epoch", report_to="none", seed=42, fp16=torch.cuda.is_available())

emo_trainer = Trainer(model=emo_model, args=emo_args,
                      train_dataset=make_dataset(emo_train), eval_dataset=make_dataset(emo_val),
                      processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=emotion_metrics)
emo_trainer.train()

# reload the best epoch (same fix as train_bert)
best = AutoModelForSequenceClassification.from_pretrained(emo_trainer.state.best_model_checkpoint)
emo_trainer.model.load_state_dict(best.state_dict(), strict=True)
del best

test_out = emo_trainer.predict(make_dataset(emo_test))
print("EMOTION TEST  accuracy =", round(test_out.metrics["test_accuracy"], 3),
      " macro_f1 =", round(test_out.metrics["test_macro_f1"], 3))

emo_trainer.save_model(MODELS_DIR + "/emotion_seed42")
tokenizer.save_pretrained(MODELS_DIR + "/emotion_seed42")
del emo_trainer, emo_model
cleanup()

## 23. Optional - publish the seed-42 models to Hugging Face Hub

Used by the Streamlit demo, which loads the models by name. Runs only when UPLOAD_TO_HUB = True in the settings cell.
The token is read from the Colab secret HF_TOKEN.

In [ ]:
UPLOADS = {
    "s1_seed42":      "sandinator/bert-stockemotions-domain-s1",
    "s2_seed42":      "sandinator/bert-stockemotions-sequential-s2",
    "s3_seed42":      "sandinator/bert-stockemotions-mixed-s3",
    "emotion_seed42": "sandinator/bert-stockemotions-emotion",
}

if UPLOAD_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    for folder_name, repo_id in UPLOADS.items():
        folder = MODELS_DIR + "/" + folder_name
        AutoModelForSequenceClassification.from_pretrained(folder).push_to_hub(repo_id)
        AutoTokenizer.from_pretrained(folder).push_to_hub(repo_id)
        print("uploaded: https://huggingface.co/" + repo_id)
else:
    print("UPLOAD_TO_HUB is False - skipped. Models stay in", MODELS_DIR)